# ASOS 일자료 기온 예측 V2

기존 44개 일자료 변수를 유지하면서 다음을 추가합니다.

- 날짜의 연중 계절성 sin/cos
- 주요 변수의 1·2·3·7·14일 lag
- 최근 3·7·14일 rolling 평균·표준편차
- 일교차·이슬점차·기압범위 등 물리 파생변수
- 절대기온이 아닌 현재 대비 미래 기온 변화량(residual) 예측
- CatBoost와 LightGBM의 검증 최적 가중 앙상블
- 기존 44개 변수 CatBoost와 동일 기간 직접 비교

Colab Secrets에 `KMA_API_KEY`를 등록하고 노트북 접근 권한을 켜주세요.

In [ ]:
!pip -q install 'catboost>=1.2,<2' 'lightgbm>=4,<5' 'scikit-learn>=1.3,<2' 'joblib>=1.3,<2'

In [ ]:
from __future__ import annotations

import json
import os
import time
from dataclasses import dataclass, asdict
from datetime import date, datetime, timedelta
from pathlib import Path
from urllib.parse import unquote

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 1. 설정과 API 키

In [ ]:
@dataclass(frozen=True)
class Config:
    station_id: str = '108'
    start_date: str = '20000101'
    end_date: str = (date.today() - timedelta(days=1)).strftime('%Y%m%d')
    train_end: str = '2018-12-31'
    valid_end: str = '2022-12-31'
    api_url: str = 'https://apis.data.go.kr/1360000/AsosDalyInfoService/getWthrDataList'
    artifact_dir: str = '/content/weather_artifacts_v2'

CFG = Config()

def load_api_key() -> str:
    key = os.getenv('KMA_API_KEY', '').strip()
    try:
        from google.colab import userdata
        key = (userdata.get('KMA_API_KEY') or key).strip()
    except (ImportError, KeyError, RuntimeError):
        pass
    if not key:
        raise RuntimeError('Colab Secrets에 KMA_API_KEY를 등록해 주세요.')
    return unquote(key)

KMA_API_KEY = load_api_key()
print('설정 완료:', asdict(CFG))


## 2. API 필드와 기존 44개 변수

In [ ]:
API_TO_KOREAN = {
    'tm': '일시', 'avgTa': '평균기온(°C)', 'minTa': '최저기온(°C)', 'maxTa': '최고기온(°C)',
    'hr1MaxRn': '1시간 최다강수량(mm)', 'sumRn': '일강수량(mm)',
    'maxInsWs': '최대 순간 풍속(m/s)', 'maxInsWsWd': '최대 순간 풍속 풍향(16방위)',
    'maxWs': '최대 풍속(m/s)', 'maxWsWd': '최대 풍속 풍향(16방위)',
    'avgWs': '평균 풍속(m/s)', 'hr24SumRws': '풍정합(100m)', 'maxWd': '최다풍향(16방위)',
    'avgTd': '평균 이슬점온도(°C)', 'minRhm': '최소 상대습도(%)', 'avgRhm': '평균 상대습도(%)',
    'avgPv': '평균 증기압(hPa)', 'avgPa': '평균 현지기압(hPa)',
    'maxPs': '최고 해면기압(hPa)', 'minPs': '최저 해면기압(hPa)', 'avgPs': '평균 해면기압(hPa)',
    'ssDur': '가조시간(hr)', 'sumSsHr': '합계 일조시간(hr)',
    'hr1MaxIcsr': '1시간 최다일사량(MJ/m2)', 'sumGsr': '합계 일사량(MJ/m2)',
    'ddMefs': '일 최심신적설(cm)', 'ddMes': '일 최심적설(cm)', 'sumDpthFhsc': '합계 3시간 신적설(cm)',
    'avgTca': '평균 전운량(1/10)', 'avgLmac': '평균 중하층운량(1/10)',
    'avgTs': '평균 지면온도(°C)', 'minTg': '최저 초상온도(°C)',
    'avgCm5Te': '평균 5cm 지중온도(°C)', 'avgCm10Te': '평균 10cm 지중온도(°C)',
    'avgCm20Te': '평균 20cm 지중온도(°C)', 'avgCm30Te': '평균 30cm 지중온도(°C)',
    'avgM05Te': '0.5m 지중온도(°C)', 'avgM10Te': '1.0m 지중온도(°C)',
    'avgM15Te': '1.5m 지중온도(°C)', 'avgM30Te': '3.0m 지중온도(°C)',
    'avgM50Te': '5.0m 지중온도(°C)', 'sumLrgEv': '합계 대형증발량(mm)',
    'sumSmlEv': '합계 소형증발량(mm)', 'n99Rn': '9-9강수(mm)', 'sumFogDur': '안개 계속시간(hr)'
}
DATE_COLUMN = '일시'
BASE_FEATURES = [name for key, name in API_TO_KOREAN.items() if key != 'tm']
TARGET_COLUMN = '목표_평균기온(°C)'
RESIDUAL_TARGET = '목표_기온변화(°C)'
assert len(BASE_FEATURES) == 44
print('기존 입력 변수:', len(BASE_FEATURES), '개')


## 3. ASOS 일자료 수집

In [ ]:
def request_page(start_dt: str, end_dt: str, page_no: int = 1, rows: int = 999) -> dict:
    params = {
        'serviceKey': KMA_API_KEY, 'pageNo': page_no, 'numOfRows': rows, 'dataType': 'JSON',
        'dataCd': 'ASOS', 'dateCd': 'DAY', 'startDt': start_dt, 'endDt': end_dt,
        'stnIds': CFG.station_id,
    }
    response = requests.get(CFG.api_url, params=params, timeout=30)
    response.raise_for_status()
    try:
        payload = response.json()
    except requests.JSONDecodeError as exc:
        raise RuntimeError(f'JSON이 아닌 응답입니다: {response.text[:300]}') from exc
    header = payload.get('response', {}).get('header', {})
    if header.get('resultCode') not in (None, '00', '0'):
        raise RuntimeError(f"API 오류: {header.get('resultCode')} {header.get('resultMsg')}")
    return payload.get('response', {}).get('body', {})

def fetch_asos_daily(start_date: str, end_date: str) -> pd.DataFrame:
    start, end = pd.Timestamp(start_date), pd.Timestamp(end_date)
    records = []
    for year in range(start.year, end.year + 1):
        chunk_start = max(start, pd.Timestamp(year=year, month=1, day=1))
        chunk_end = min(end, pd.Timestamp(year=year, month=12, day=31))
        page = 1
        while True:
            body = request_page(chunk_start.strftime('%Y%m%d'), chunk_end.strftime('%Y%m%d'), page)
            items = body.get('items') or {}
            rows = items.get('item') or []
            rows = [rows] if isinstance(rows, dict) else rows
            records.extend(rows)
            if page * 999 >= int(body.get('totalCount') or 0):
                break
            page += 1
        print(year, '완료:', len(records), '누적 행')
        time.sleep(0.1)
    if not records:
        raise RuntimeError('수집된 ASOS 일자료가 없습니다.')
    frame = pd.DataFrame(records).rename(columns=API_TO_KOREAN)
    missing = sorted(set([DATE_COLUMN, *BASE_FEATURES]) - set(frame.columns))
    if missing:
        raise RuntimeError(f'API 응답에 필요한 필드가 없습니다: {missing}')
    frame = frame[[DATE_COLUMN, *BASE_FEATURES]].copy()
    frame[DATE_COLUMN] = pd.to_datetime(frame[DATE_COLUMN])
    return frame.drop_duplicates(DATE_COLUMN).sort_values(DATE_COLUMN).reset_index(drop=True)

CACHE_PATH = Path(f'/content/asos_daily_{CFG.station_id}_{CFG.start_date}_{CFG.end_date}.csv')
if CACHE_PATH.exists():
    raw_df = pd.read_csv(CACHE_PATH, parse_dates=[DATE_COLUMN])
    print('캐시 사용:', CACHE_PATH)
else:
    raw_df = fetch_asos_daily(CFG.start_date, CFG.end_date)
    raw_df.to_csv(CACHE_PATH, index=False, encoding='utf-8-sig')
print('원본 크기:', raw_df.shape)


## 4. 인과적 전처리와 V2 파생변수
미래값을 사용하지 않도록 모든 lag와 rolling은 현재 또는 과거 자료만 사용합니다. 실제 한파·폭염을 보존하기 위해 V1의 분위수 clipping은 제거했습니다.

In [ ]:
ZERO_FILL_FEATURES = [
    c for c in BASE_FEATURES
    if any(token in c for token in ('강수', '적설', '안개', '증발량'))
]
TEMPORAL_FEATURES = [
    '평균기온(°C)', '최저기온(°C)', '최고기온(°C)', '평균 이슬점온도(°C)',
    '평균 상대습도(%)', '평균 현지기압(hPa)', '평균 해면기압(hPa)', '평균 풍속(m/s)',
    '일강수량(mm)', '평균 전운량(1/10)', '평균 지면온도(°C)',
    '평균 5cm 지중온도(°C)', '평균 30cm 지중온도(°C)',
    '합계 일조시간(hr)', '합계 일사량(MJ/m2)'
]
LAGS = (1, 2, 3, 7, 14)
ROLLING_WINDOWS = (3, 7, 14)

def clean_raw_daily(frame: pd.DataFrame) -> pd.DataFrame:
    data = frame[[DATE_COLUMN, *BASE_FEATURES]].copy().sort_values(DATE_COLUMN)
    for column in BASE_FEATURES:
        data[column] = pd.to_numeric(data[column], errors='coerce')
    data[ZERO_FILL_FEATURES] = data[ZERO_FILL_FEATURES].fillna(0.0)
    # 시간순 ffill만 사용하므로 미래 관측값이 과거로 유입되지 않습니다.
    data[BASE_FEATURES] = data[BASE_FEATURES].ffill()
    return data

def build_v2_features(clean: pd.DataFrame) -> pd.DataFrame:
    data = clean.copy()
    day_of_year = data[DATE_COLUMN].dt.dayofyear
    engineered = {
        '연중일_sin': np.sin(2 * np.pi * day_of_year / 365.25),
        '연중일_cos': np.cos(2 * np.pi * day_of_year / 365.25),
        '일교차(°C)': data['최고기온(°C)'] - data['최저기온(°C)'],
        '기온_이슬점차(°C)': data['평균기온(°C)'] - data['평균 이슬점온도(°C)'],
        '해면기압범위(hPa)': data['최고 해면기압(hPa)'] - data['최저 해면기압(hPa)'],
        '지면_대기온도차(°C)': data['평균 지면온도(°C)'] - data['평균기온(°C)'],
        '30cm지중_대기온도차(°C)': data['평균 30cm 지중온도(°C)'] - data['평균기온(°C)'],
        '일조율': (data['합계 일조시간(hr)'] / data['가조시간(hr)'].replace(0, np.nan)).clip(0, 1),
    }
    for column in TEMPORAL_FEATURES:
        for lag in LAGS:
            engineered[f'{column}__lag{lag}'] = data[column].shift(lag)
        engineered[f'{column}__delta1'] = data[column] - data[column].shift(1)
        engineered[f'{column}__delta3'] = data[column] - data[column].shift(3)
        for window in ROLLING_WINDOWS:
            history = data[column].rolling(window=window, min_periods=window)
            engineered[f'{column}__mean{window}'] = history.mean()
            engineered[f'{column}__std{window}'] = history.std()
    return pd.concat([data, pd.DataFrame(engineered, index=data.index)], axis=1)

clean_df = clean_raw_daily(raw_df)
feature_df = build_v2_features(clean_df)
target_lookup = clean_df[[DATE_COLUMN, '평균기온(°C)']].copy()
target_lookup[DATE_COLUMN] = target_lookup[DATE_COLUMN] - pd.Timedelta(days=2)
target_lookup = target_lookup.rename(columns={'평균기온(°C)': TARGET_COLUMN})
dataset = feature_df.merge(target_lookup, on=DATE_COLUMN, how='left')
dataset[RESIDUAL_TARGET] = dataset[TARGET_COLUMN] - dataset['평균기온(°C)']
dataset = dataset.dropna(subset=[TARGET_COLUMN, RESIDUAL_TARGET])
V2_FEATURES = [c for c in feature_df.columns if c != DATE_COLUMN]
print('기본 변수:', len(BASE_FEATURES), 'V2 전체 변수:', len(V2_FEATURES), '데이터:', len(dataset))


## 5. 시간 분할과 학습 구간 전용 결측치 처리

In [ ]:
train_df = dataset[dataset[DATE_COLUMN] <= CFG.train_end].copy()
valid_df = dataset[(dataset[DATE_COLUMN] > CFG.train_end) & (dataset[DATE_COLUMN] <= CFG.valid_end)].copy()
test_df = dataset[dataset[DATE_COLUMN] > CFG.valid_end].copy()
if min(len(train_df), len(valid_df), len(test_df)) == 0:
    raise RuntimeError('분할 중 비어 있는 구간이 있습니다.')

class MedianPreprocessor:
    def __init__(self, feature_columns):
        self.feature_columns = list(feature_columns)

    def fit(self, frame):
        numeric = frame[self.feature_columns].apply(pd.to_numeric, errors='coerce')
        self.medians_ = numeric.median().fillna(0.0)
        self.missing_features_ = [c for c in self.feature_columns if numeric[c].isna().any()]
        self.output_columns_ = self.feature_columns + [f'{c}__missing' for c in self.missing_features_]
        return self

    def transform(self, frame):
        numeric = frame[self.feature_columns].apply(pd.to_numeric, errors='coerce')
        indicators = {f'{c}__missing': numeric[c].isna().astype('int8') for c in self.missing_features_}
        filled = numeric.fillna(self.medians_)
        if indicators:
            filled = pd.concat([filled, pd.DataFrame(indicators, index=filled.index)], axis=1)
        return filled[self.output_columns_].astype('float32')

v2_preprocessor = MedianPreprocessor(V2_FEATURES).fit(train_df)
X_train = v2_preprocessor.transform(train_df)
X_valid = v2_preprocessor.transform(valid_df)
X_test = v2_preprocessor.transform(test_df)
y_train_res = train_df[RESIDUAL_TARGET].astype('float32')
y_valid_res = valid_df[RESIDUAL_TARGET].astype('float32')
y_test = test_df[TARGET_COLUMN].astype('float32')
current_valid = valid_df['평균기온(°C)'].to_numpy(dtype='float32')
current_test = test_df['평균기온(°C)'].to_numpy(dtype='float32')
print({'train': X_train.shape, 'valid': X_valid.shape, 'test': X_test.shape})


## 6. 공정 비교용 V1 CatBoost
동일한 분할과 원자료에서 기존 44개 변수만 사용하는 모델을 다시 학습합니다.

In [ ]:
def metrics(y_true, y_pred):
    return {
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'rmse': float(mean_squared_error(y_true, y_pred) ** 0.5),
        'r2': float(r2_score(y_true, y_pred)),
    }

base_preprocessor = MedianPreprocessor(BASE_FEATURES).fit(train_df)
X_train_base = base_preprocessor.transform(train_df)
X_valid_base = base_preprocessor.transform(valid_df)
X_test_base = base_preprocessor.transform(test_df)
y_train = train_df[TARGET_COLUMN].astype('float32')
y_valid = valid_df[TARGET_COLUMN].astype('float32')

v1_model = CatBoostRegressor(
    loss_function='MAE', eval_metric='MAE', iterations=3000, learning_rate=0.025, depth=8,
    l2_leaf_reg=5.0, random_strength=0.5, bagging_temperature=0.5,
    random_seed=RANDOM_SEED, allow_writing_files=False, verbose=200,
)
v1_model.fit(X_train_base, y_train, eval_set=(X_valid_base, y_valid), use_best_model=True, early_stopping_rounds=200)
v1_test_pred = v1_model.predict(X_test_base)
persistence_pred = current_test
print('Persistence:', metrics(y_test, persistence_pred))
print('V1 CatBoost:', metrics(y_test, v1_test_pred))


## 7. V2 Residual CatBoost와 LightGBM

In [ ]:
cat_v2 = CatBoostRegressor(
    loss_function='MAE', eval_metric='MAE', iterations=4000, learning_rate=0.02, depth=8,
    l2_leaf_reg=7.0, random_strength=0.3, bagging_temperature=0.4,
    random_seed=RANDOM_SEED, allow_writing_files=False, verbose=200,
)
cat_v2.fit(X_train, y_train_res, eval_set=(X_valid, y_valid_res), use_best_model=True, early_stopping_rounds=250)

lgb_v2 = lgb.LGBMRegressor(
    objective='regression_l1', n_estimators=4000, learning_rate=0.02, num_leaves=31,
    max_depth=-1, min_child_samples=30, subsample=0.85, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, random_state=RANDOM_SEED, n_jobs=-1, verbosity=-1,
)
lgb_v2.fit(
    X_train, y_train_res, eval_set=[(X_valid, y_valid_res)], eval_metric='mae',
    callbacks=[lgb.early_stopping(250, verbose=False), lgb.log_evaluation(200)],
)

cat_valid_pred = current_valid + cat_v2.predict(X_valid)
cat_test_pred = current_test + cat_v2.predict(X_test)
lgb_valid_pred = current_valid + lgb_v2.predict(X_valid)
lgb_test_pred = current_test + lgb_v2.predict(X_test)
print('V2 CatBoost:', metrics(y_test, cat_test_pred))
print('V2 LightGBM:', metrics(y_test, lgb_test_pred))


## 8. 검증 데이터로 앙상블 가중치 선택 및 최종 평가

In [ ]:
weights = np.linspace(0.0, 1.0, 101)
validation_scores = [
    mean_absolute_error(y_valid, weight * cat_valid_pred + (1 - weight) * lgb_valid_pred)
    for weight in weights
]
best_weight = float(weights[int(np.argmin(validation_scores))])
ensemble_pred = best_weight * cat_test_pred + (1 - best_weight) * lgb_test_pred

scores = pd.DataFrame([
    {'model': 'Persistence', **metrics(y_test, persistence_pred)},
    {'model': 'V1 CatBoost 44개', **metrics(y_test, v1_test_pred)},
    {'model': 'V2 Residual CatBoost', **metrics(y_test, cat_test_pred)},
    {'model': 'V2 Residual LightGBM', **metrics(y_test, lgb_test_pred)},
    {'model': f'V2 Ensemble cat={best_weight:.2f}', **metrics(y_test, ensemble_pred)},
]).sort_values('mae')
display(scores)

result = test_df[[DATE_COLUMN]].copy()
result['target_date'] = result[DATE_COLUMN] + pd.Timedelta(days=2)
result['actual'] = y_test.to_numpy()
result['prediction'] = ensemble_pred
result['absolute_error'] = np.abs(result['actual'] - result['prediction'])
result['season'] = result['target_date'].dt.month.map({12:'겨울',1:'겨울',2:'겨울',3:'봄',4:'봄',5:'봄',6:'여름',7:'여름',8:'여름',9:'가을',10:'가을',11:'가을'})
display(result.groupby('season', sort=False)['absolute_error'].agg(['count', 'mean', 'median', 'max']))
display(result.nlargest(20, 'absolute_error'))

plt.figure(figsize=(16, 5))
plt.plot(result['target_date'], result['actual'], label='actual', linewidth=1)
plt.plot(result['target_date'], result['prediction'], label='V2 ensemble', linewidth=1)
plt.title('V2 daily average temperature prediction')
plt.ylabel('°C')
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 9. 변수 중요도

In [ ]:
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': cat_v2.get_feature_importance(),
}).sort_values('importance', ascending=False).head(30)
display(importance)
plt.figure(figsize=(10, 9))
plt.barh(importance['feature'][::-1], importance['importance'][::-1])
plt.title('V2 CatBoost top 30 feature importance')
plt.tight_layout()
plt.show()


## 10. 모델 산출물 저장

In [ ]:
artifact_dir = Path(CFG.artifact_dir)
artifact_dir.mkdir(parents=True, exist_ok=True)
model_version = f"ensemble-asos-daily-v2-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
cat_v2.save_model(artifact_dir / 'catboost_residual_v2.cbm')
joblib.dump(lgb_v2, artifact_dir / 'lightgbm_residual_v2.joblib')
joblib.dump(v2_preprocessor, artifact_dir / 'preprocessor_v2.joblib')
(artifact_dir / 'feature_columns_v2.json').write_text(
    json.dumps(V2_FEATURES, ensure_ascii=False, indent=2), encoding='utf-8'
)
metadata = {
    'model_version': model_version, 'station_id': CFG.station_id,
    'forecast_offset_days': 2, 'base_feature_count': len(BASE_FEATURES),
    'engineered_feature_count': len(V2_FEATURES), 'model_input_count': X_train.shape[1],
    'catboost_weight': best_weight, 'lightgbm_weight': 1 - best_weight,
    'scores': scores.to_dict(orient='records'), 'config': asdict(CFG),
}
(artifact_dir / 'model_metadata_v2.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
)
result.to_csv(artifact_dir / 'test_predictions_v2.csv', index=False, encoding='utf-8-sig')
print('저장 완료:', artifact_dir)
print('\n'.join(str(path) for path in sorted(artifact_dir.iterdir())))


## 11. 최근 30일 자료로 내일 평균기온 예측
lag와 rolling 변수 생성을 위해 하루가 아니라 최근 30일 자료를 가져옵니다.

In [ ]:
observation_date = pd.Timestamp(date.today() - timedelta(days=1))
history_start = observation_date - pd.Timedelta(days=30)
latest_raw = fetch_asos_daily(history_start.strftime('%Y%m%d'), observation_date.strftime('%Y%m%d'))
latest_clean = clean_raw_daily(latest_raw)
latest_features = build_v2_features(latest_clean)
latest_row = latest_features[latest_features[DATE_COLUMN] == observation_date]
if len(latest_row) != 1:
    raise RuntimeError('예측 기준일 자료를 찾을 수 없습니다.')
latest_X = v2_preprocessor.transform(latest_row)
current_temperature = float(latest_row['평균기온(°C)'].iloc[0])
cat_prediction = current_temperature + float(cat_v2.predict(latest_X)[0])
lgb_prediction = current_temperature + float(lgb_v2.predict(latest_X)[0])
final_prediction = best_weight * cat_prediction + (1 - best_weight) * lgb_prediction
prediction_payload = {
    'station_id': CFG.station_id,
    'observation_date': str(observation_date.date()),
    'predicted_for_date': str((observation_date + pd.Timedelta(days=2)).date()),
    'predicted_avg_temperature': round(final_prediction, 2),
    'model_version': model_version,
}
print(json.dumps(prediction_payload, ensure_ascii=False, indent=2))

# 런타임 종료 전에 필요하면 아래 주석을 해제해 산출물을 다운로드하세요.
# !zip -qr /content/weather_artifacts_v2.zip /content/weather_artifacts_v2
# from google.colab import files
# files.download('/content/weather_artifacts_v2.zip')
